# 03 - Seleção e Treinamento de Modelos

## Passos Mágicos - Predição de Risco de Defasagem Escolar

Este notebook compara diferentes algoritmos de ML e seleciona o melhor modelo.

### Objetivos:
1. Treinar múltiplos modelos
2. Comparar performance (foco em Recall)
3. Otimizar hiperparâmetros
4. Selecionar e salvar melhor modelo

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.preprocessing import DataLoader, DataCleaner
from src.features import FeatureEngineer, FeatureSelector
from src.models import ModelTrainer, ModelEvaluator
from src.models.train import train_multiple_models
from src.models.evaluate import compare_models
from src.config import TRAIN_CONFIG, RISCO_LABELS

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Preparação dos Dados

In [ ]:
# Pipeline completo de preparação
loader = DataLoader()
df = loader.load_and_unify()

cleaner = DataCleaner()
df = cleaner.fit_transform(df)

engineer = FeatureEngineer()
df = engineer.create_all_features(df)

# Preparar X e y
exclude_cols = ['RISCO_DEFASAGEM', 'NIVEL_RISCO', 'RA', 'NOME', 'ANO_PEDE', 
                'PEDRA', 'GENERO', 'INSTITUICAO_ENSINO_ALUNO', 'TURMA', 'FAIXA_ETARIA', 'INDE_CATEGORIA']

feature_cols = [col for col in df.columns 
                if col not in exclude_cols 
                and df[col].dtype in ['int64', 'float64', 'int32', 'float32']]

X = df[feature_cols].copy()
y = df['RISCO_DEFASAGEM'].copy()

# Seleção de features
selector = FeatureSelector(method='importance', n_features=15)
X = selector.fit_transform(X, y)

print(f"Features: {X.shape[1]}")
print(f"Amostras: {len(X)}")

In [ ]:
# Split treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TRAIN_CONFIG['test_size'],
    random_state=TRAIN_CONFIG['random_state'],
    stratify=y
)

print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")
print(f"\nDistribuição no treino:")
print(y_train.value_counts(normalize=True).round(3))

## 2. Treinamento de Múltiplos Modelos

In [ ]:
# Treinar modelos
model_types = ['logistic_regression', 'random_forest', 'gradient_boosting']

trainers = {}
for model_type in model_types:
    print(f"\nTreinando {model_type}...")
    trainer = ModelTrainer(model_type=model_type, scoring='recall')
    trainer.train(X_train, y_train, optimize_hyperparams=True)
    trainers[model_type] = trainer
    print(f"  CV Score: {trainer.cv_results.get('best_score', 'N/A'):.4f}")

## 3. Comparação de Modelos

In [ ]:
# Avaliar modelos no conjunto de teste
results = []

for name, trainer in trainers.items():
    evaluator = ModelEvaluator(trainer.pipeline)
    metrics = evaluator.evaluate(X_test, y_test)
    metrics['model'] = name
    results.append(metrics)

results_df = pd.DataFrame(results).set_index('model')
results_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4)

In [ ]:
# Visualização comparativa
fig, ax = plt.subplots(figsize=(10, 6))

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
results_df[metrics_to_plot].plot(kind='bar', ax=ax, width=0.8)

plt.title('Comparação de Modelos')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Análise do Melhor Modelo

In [ ]:
# Selecionar melhor modelo (baseado em Recall)
best_model_name = results_df['recall'].idxmax()
best_trainer = trainers[best_model_name]

print(f"Melhor modelo: {best_model_name}")
print(f"Recall: {results_df.loc[best_model_name, 'recall']:.4f}")

In [ ]:
# Matriz de confusão do melhor modelo
evaluator = ModelEvaluator(best_trainer.pipeline)
evaluator.evaluate(X_test, y_test)

cm = evaluator.get_confusion_matrix()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['BAIXO', 'MÉDIO', 'ALTO'],
            yticklabels=['BAIXO', 'MÉDIO', 'ALTO'])
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title(f'Matriz de Confusão - {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Relatório de classificação
print(evaluator.get_classification_report())

## 5. Salvar Modelo Final

In [ ]:
# Salvar melhor modelo
model_path = best_trainer.save()
print(f"Modelo salvo em: {model_path}")

## 6. Conclusões

### Justificativa da Métrica (Recall):

No contexto educacional, é **crítico identificar todos os alunos em risco de defasagem**. 
Um falso negativo (aluno em risco não identificado) tem consequências mais graves que um falso positivo.

Por isso, priorizamos **Recall alto** mesmo com alguma redução em Precision.

### Resultados:
- Modelo selecionado com base no melhor Recall
- Modelo salvo e pronto para deploy
- API disponível para predições em produção